# Deep Learning Architectures for Pneumonia Detection: Comprehensive XAI Analysis

**XAI Analysis Notebook**

**Description:** This notebook provides comprehensive explainable AI (XAI) analysis for trained pneumonia detection models using GradCAM, LIME, and SHAP. It compares XAI methods across different architectures and resolutions, analyzes misclassifications, and provides similarity metrics between explanation methods.

**Main Features:**
- Comparison across architectures (AlexNet, ResNet18, ResNet50, DenseNet169)
- Resolution comparison (28px, 64px, 128px)  
- XAI method comparison (GradCAM, LIME, SHAP)
- Misclassification analysis
- Memory-efficient single-model loading
- Comprehensive similarity metrics

**Notebook Structure:**
- **Step 1:** Load or Generate Predictions
- **Step 1.5:** Load Sample Images for XAI Analysis  
- **Step 2:** Generate XAI Analysis for Each Model
- **Step 3:** Individual Model XAI Visualizations
- **Step 4:** Comprehensive XAI Comparisons
- **Step 5:** Misclassification Analysis
- **Step 6:** XAI Similarity Metrics
- **Step 7:** Generate Comprehensive Analysis Report

**Authors:**
- Rafaela Abrunhosa, 107658
- Miguel Pinto, 107449

In [35]:
# Clone the repository
# !git clone https://github.com/mariaabr/CAA_Project2.git
!rm -rf CAA_Project2
!git clone -b xai2 https://github.com/mariaabr/CAA_Project2.git

# Mount Google Drive
!fusermount -u /content/drive 2>/dev/null
!rm -rf /content/drive

from google.colab import drive
drive.mount('/content/drive')

# Copy the data and models from Drive to the repository
!cp -r /content/drive/MyDrive/data /content/CAA_Project2/
!cp -r /content/drive/MyDrive/models /content/CAA_Project2/output/

# Enter the notebooks folder - session restarting point
%cd CAA_Project2/notebooks

Cloning into 'CAA_Project2'...
remote: Enumerating objects: 505, done.
remote: Counting objects: 100% (178/178), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 505 (delta 90), reused 140 (delta 62), pack-reused 327 (from 1)
Receiving objects: 100% (505/505), 21.48 MiB | 25.14 MiB/s, done.
Resolving deltas: 100% (287/287), done.
Mounted at /content/drive
/content/CAA_Project2/notebooks/CAA_Project2/notebooks/CAA_Project2/notebooks/CAA_Project2/notebooks/CAA_Project2/notebooks


In [36]:
!ls -l /content/CAA_Project2/output/models/alexnet_augmented_b96_lr0.001_dr0.5_pneumonia_model.keras

-rw------- 1 root root 355044505 Jun 22 12:37 /content/CAA_Project2/output/models/alexnet_augmented_b96_lr0.001_dr0.5_pneumonia_model.keras


In [37]:
# turn script .sh executable
!chmod +x /content/CAA_Project2/scripts/generate_all_predictions.sh

# execute script
!cd /content/CAA_Project2/scripts/ && ./generate_all_predictions.sh

XAI Prediction Generation Script
Script directory: /content/CAA_Project2/scripts
Models file: /content/CAA_Project2/scripts/../models.txt
Python script: /content/CAA_Project2/scripts/generate_predictions.py

Reading models from: /content/CAA_Project2/scripts/../models.txt

Processing model: alexnet_augmented_b96_lr0.001_dr0.5_pneumonia_model.keras
----------------------------------------
2025-06-22 12:38:15.795831: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750595895.845246   26044 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750595895.855757   26044 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Processing: alexnet_augmented_b96_lr0.001_dr0.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# External Libraries

In [ ]:
# Import utility libraries
import os
import sys
import numpy as np
import random
import gc
import pickle
import json
sys.path.append('..')  # Add parent directory to path

# Import visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Import deep learning libraries
import tensorflow as tf
from tensorflow import keras

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# %pip install lime
# %pip install shap
# %pip install tf_explain

# Import custom utilities
from utils.xai_utils import *

print(f"TensorFlow version: {tf.__version__}")
print("XAI utilities loaded successfully!")
print("Visualization libraries loaded!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
TensorFlow version: 2.18.0
XAI utilities loaded successfully!
Visualization libraries loaded!


# Configuration

In [ ]:
# Configuration
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Paths
DATA_PATH = '../data/'
MODEL_PATH = '../output/models/'
PREDICTIONS_PATH = '../output/predictions/'
OUTPUT_PATH = '../output/xai_results/'

# XAI Configuration
NUM_SAMPLES_TO_ANALYZE = 20  # Number of images to analyze per class (balanced)
NUM_LIME_SAMPLES = 5  # Limit LIME to fewer samples (it's slow)
NUM_SHAP_SAMPLES = 10  # Limit SHAP samples for memory efficiency

# Analysis modes
SKIP_PREDICTION_GENERATION = True  # Set to True if predictions are already generated
PERFORM_GRADCAM = True
PERFORM_LIME = True
PERFORM_SHAP = True  # Set to False if memory is limited

# All available models from models.txt
# Read models from models.txt
with open('../models.txt', 'r') as f:
    ALL_AVAILABLE_MODELS = [line.strip() for line in f.readlines() if line.strip() and not line.startswith('#')]

# Models to analyze (you can subset this for testing)
MODELS_TO_ANALYZE = ALL_AVAILABLE_MODELS

# Create output directories
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(f"{OUTPUT_PATH}/predictions", exist_ok=True)
os.makedirs(f"{OUTPUT_PATH}/comparisons", exist_ok=True)
os.makedirs(f"{OUTPUT_PATH}/individual_results", exist_ok=True)

print(f"Configuration loaded!")
print(f"Will analyze {len(MODELS_TO_ANALYZE)} models")
print(f"Output directory: {OUTPUT_PATH}")
print(f"Skip prediction generation: {SKIP_PREDICTION_GENERATION}")
print(f"Methods enabled: GradCAM={PERFORM_GRADCAM}, LIME={PERFORM_LIME}, SHAP={PERFORM_SHAP}")

# Check existing prediction files if skipping generation
if SKIP_PREDICTION_GENERATION:
    print(f"\nChecking for existing prediction files in: {PREDICTIONS_PATH}")
    prediction_files = []
    if os.path.exists(PREDICTIONS_PATH):
        for file in os.listdir(PREDICTIONS_PATH):
            if file.endswith('.pkl'):
                prediction_files.append(file)
                print(f"✓ Found: {file}")
    print(f"Found {len(prediction_files)} prediction files")
else:
    # Check which models exist for prediction generation
    existing_models = []
    for model_file in MODELS_TO_ANALYZE:
        model_path = os.path.join(MODEL_PATH, model_file)
        if os.path.exists(model_path):
            existing_models.append(model_file)
            print(f"✓ {model_file}")
        else:
            print(f"✗ {model_file} (missing)")
    print(f"\nFound {len(existing_models)} existing models out of {len(MODELS_TO_ANALYZE)}")

Configuration loaded!
Will analyze 12 models
Output directory: ../output/xai_results/
Skip prediction generation: True
Methods enabled: GradCAM=True, LIME=True, SHAP=True

Checking for existing prediction files in: ../output/predictions/
Found 0 prediction files


# Step 1: Load or Generate Predictions

In [ ]:
# Step 1: Load or Generate Predictions

print("="*80)
print("STEP 1: LOAD OR GENERATE PREDICTIONS")
print("="*80)

predictions_files = {}

if SKIP_PREDICTION_GENERATION:
    print("Loading existing prediction files...")

    # Load existing prediction files
    if os.path.exists(PREDICTIONS_PATH):
        for file in os.listdir(PREDICTIONS_PATH):
            if file.endswith('.pkl'):
                try:
                    # Load prediction data to get model info
                    with open(os.path.join(PREDICTIONS_PATH, file), 'rb') as f:
                        pred_data = pickle.load(f)

                    model_info = pred_data['model_info']
                    key = (model_info['architecture'], model_info['resolution'])
                    predictions_files[key] = os.path.join(PREDICTIONS_PATH, file)

                    print(f"✓ Loaded: {model_info['architecture']} ({model_info['resolution']}px) - Accuracy: {pred_data['accuracy']:.3f}")

                except Exception as e:
                    print(f"✗ Failed to load {file}: {e}")

    print(f"\n✓ Loaded {len(predictions_files)} prediction files")

else:
    print("Generating predictions for selected models...")

    # This step processes one model at a time to save memory
    for model_file in MODELS_TO_ANALYZE:
        if model_file not in existing_models:
            continue

        print(f"\nProcessing: {model_file}")

        # Parse model information
        model_info = parse_model_info(model_file)
        print(f"Architecture: {model_info['architecture']}")
        print(f"Resolution: {model_info['resolution']}px")
        print(f"Is tuned: {model_info['is_tuned']}")

        # Check if predictions already exist
        pred_file = f"{PREDICTIONS_PATH}/{model_info['architecture']}_{model_info['resolution']}px_predictions.pkl"
        if os.path.exists(pred_file):
            print(f"✓ Predictions already exist: {pred_file}")
            predictions_files[(model_info['architecture'], model_info['resolution'])] = pred_file
            continue

        # Load test data for this resolution
        test_images, test_labels = load_test_data(DATA_PATH, model_info['resolution'])
        print(f"Test data loaded: {test_images.shape}")

        # Load and predict with model
        model_path = os.path.join(MODEL_PATH, model_file)
        model = tf.keras.models.load_model(model_path)
        print(f"Model loaded successfully!")

        # Save predictions
        pred_file = save_model_predictions(model, test_images, test_labels, model_info, PREDICTIONS_PATH)
        predictions_files[(model_info['architecture'], model_info['resolution'])] = pred_file

        # Clear memory
        del model
        del test_images, test_labels
        gc.collect()
        tf.keras.backend.clear_session()

        print(f"✓ Predictions saved and memory cleared")

print(f"\n✓ Prediction loading/generation completed!")
print(f"Available predictions: {len(predictions_files)}")

# Display available predictions summary
print(f"\nAvailable Predictions Summary:")
print("-" * 40)
for (arch, res), file_path in predictions_files.items():
    print(f"{arch:12} {res:3}px: {os.path.basename(file_path)}")

if not predictions_files:
    print("⚠️  No prediction files available. Please either:")
    print("   1. Set SKIP_PREDICTION_GENERATION = False to generate predictions")
    print("   2. Run the bash script: scripts/generate_all_predictions.sh")
    print("   3. Generate predictions manually using scripts/generate_predictions.py")

STEP 1: LOAD OR GENERATE PREDICTIONS
Loading existing prediction files...

✓ Loaded 0 prediction files

✓ Prediction loading/generation completed!
Available predictions: 0

Available Predictions Summary:
----------------------------------------
⚠️  No prediction files available. Please either:
   1. Set SKIP_PREDICTION_GENERATION = False to generate predictions
   2. Run the bash script: scripts/generate_all_predictions.sh
   3. Generate predictions manually using scripts/generate_predictions.py


# Step 1.5: Load Sample Images for XAI Analysis

In [ ]:
print("="*80)
print("STEP 1.5: LOADING SAMPLE IMAGES FOR XAI ANALYSIS")
print("="*80)

# Load sample images for each resolution
sample_images = {}
sample_labels = {}
sample_indices = {}

resolutions = list(set([res for (arch, res) in predictions_files.keys()]))
print(f"Resolutions to analyze: {resolutions}")

for resolution in resolutions:
    print(f"\nLoading {resolution}px samples...")

    # Load test data
    test_images, test_labels = load_test_data(DATA_PATH, resolution)

    # Select samples
    selected_images, selected_labels, selected_indices = select_sample_images(
        test_images, test_labels, NUM_SAMPLES_TO_ANALYZE, RANDOM_SEED
    )

    sample_images[resolution] = selected_images
    sample_labels[resolution] = selected_labels
    sample_indices[resolution] = selected_indices

    print(f"✓ Selected {len(selected_images)} samples for {resolution}px")

    # Clear full test data to save memory
    del test_images, test_labels

print(f"\n✓ Sample images loaded for all resolutions!")

# Display sample selection summary
class_names = ['Normal', 'Pneumonia']
for resolution in resolutions:
    print(f"\n{resolution}px samples:")
    for i, class_name in enumerate(class_names):
        count = np.sum(sample_labels[resolution] == i)
        print(f"  {class_name}: {count} images")

STEP 1.5: LOADING SAMPLE IMAGES FOR XAI ANALYSIS
Resolutions to analyze: []

✓ Sample images loaded for all resolutions!


# Step 2: Generate XAI Analysis for Each Model

In [ ]:
# Step 2: Generate XAI Analysis for Each Model

print("="*80)
print("STEP 2: GENERATING XAI ANALYSIS FOR EACH MODEL")
print("="*80)

if not predictions_files:
    print("⚠️  No predictions available. Skipping XAI analysis.")
    xai_results = {}
else:
    # This will store all XAI results
    xai_results = {}

    for (architecture, resolution), pred_file in predictions_files.items():
        print(f"\n{'='*50}")
        print(f"Analyzing: {architecture} ({resolution}px)")
        print(f"{'='*50}")

        # Load prediction data
        with open(pred_file, 'rb') as f:
            pred_data = pickle.load(f)

        model_info = pred_data['model_info']
        model_file = model_info['filename']

        # Load model
        model_path = os.path.join(MODEL_PATH, model_file)
        if not os.path.exists(model_path):
            print(f"✗ Model file not found: {model_file}")
            continue

        model = tf.keras.models.load_model(model_path)
        print(f"✓ Model loaded: {architecture} ({resolution}px)")

        # Get sample images for this resolution
        if resolution not in sample_images:
            print(f"✗ No sample images available for {resolution}px")
            del model
            gc.collect()
            tf.keras.backend.clear_session()
            continue

        images = sample_images[resolution]
        labels = sample_labels[resolution]

        # Get model predictions for samples
        predictions = model.predict(images, verbose=0)
        pred_classes = np.argmax(predictions, axis=1)
        pred_probs = np.max(predictions, axis=1)

        accuracy = np.sum(pred_classes == labels) / len(labels)
        print(f"Sample predictions accuracy: {np.sum(pred_classes == labels)}/{len(labels)} ({accuracy*100:.1f}%)")

        # Initialize results structure
        model_results = {
            'pred_classes': pred_classes,
            'true_classes': labels,
            'pred_probs': pred_probs,
            'gradcam': None,
            'lime': None,
            'shap': None
        }

        # 1. GradCAM Analysis
        if PERFORM_GRADCAM:
            print("\n1. Applying GradCAM...")
            try:
                gradcam_results, used_layer = apply_gradcam_analysis(model, images, pred_classes)
                model_results['gradcam'] = gradcam_results

                success_count = sum(1 for x in gradcam_results if x is not None)
                print(f"✓ GradCAM completed using layer: {used_layer} ({success_count}/{len(images)} successful)")
            except Exception as e:
                print(f"✗ GradCAM failed: {e}")
                model_results['gradcam'] = [None] * len(images)
        else:
            print("\n1. Skipping GradCAM (disabled in config)")
            model_results['gradcam'] = [None] * len(images)

        # 2. LIME Analysis (limited samples)
        if PERFORM_LIME:
            print(f"\n2. Applying LIME (first {NUM_LIME_SAMPLES} samples)...")
            try:
                lime_images = images[:NUM_LIME_SAMPLES]
                lime_results = apply_lime_analysis(model, lime_images)
                # Extend to full length with None values
                full_lime_results = lime_results + [None] * (len(images) - len(lime_results))
                model_results['lime'] = full_lime_results

                success_count = sum(1 for x in lime_results if x is not None)
                print(f"✓ LIME completed for {success_count}/{len(lime_images)} samples")
            except Exception as e:
                print(f"✗ LIME failed: {e}")
                model_results['lime'] = [None] * len(images)
        else:
            print("\n2. Skipping LIME (disabled in config)")
            model_results['lime'] = [None] * len(images)

        # 3. SHAP Analysis
        if PERFORM_SHAP:
            print(f"\n3. Applying SHAP (first {NUM_SHAP_SAMPLES} samples)...")
            try:
                shap_images = images[:NUM_SHAP_SAMPLES]
                shap_values = apply_shap_analysis(model, shap_images)
                model_results['shap'] = shap_values
                if shap_values is not None:
                    print("✓ SHAP completed")
                else:
                    print("✗ SHAP failed")
            except Exception as e:
                print(f"✗ SHAP failed: {e}")
                model_results['shap'] = None
        else:
            print("\n3. Skipping SHAP (disabled in config)")
            model_results['shap'] = None

        # Store results using a tuple key (architecture, resolution)
        xai_results[(architecture, resolution)] = model_results

        # Save individual results
        individual_results_file = f"{OUTPUT_PATH}/individual_results/{architecture}_{resolution}px_xai_results.pkl"
        with open(individual_results_file, 'wb') as f:
            pickle.dump(model_results, f)
        print(f"✓ Individual results saved: {individual_results_file}")

        # Clear memory
        del model
        gc.collect()
        tf.keras.backend.clear_session()

        print(f"✓ XAI analysis completed and memory cleared")

print(f"\n✓ All XAI analyses completed!")
print(f"Analyzed models: {len(xai_results)}")

# Summary of XAI results
if xai_results:
    print(f"\nXAI Analysis Summary:")
    print("-" * 30)
    for (arch, res), results in xai_results.items():
        gradcam_success = sum(1 for x in results['gradcam'] if x is not None) if results['gradcam'] else 0
        lime_success = sum(1 for x in results['lime'] if x is not None) if results['lime'] else 0
        shap_success = 1 if results['shap'] is not None else 0

        print(f"{arch:12} {res:3}px: GradCAM={gradcam_success:2}, LIME={lime_success:2}, SHAP={shap_success}")
else:
    print("\n⚠️  No XAI results generated.")

STEP 2: GENERATING XAI ANALYSIS FOR EACH MODEL
⚠️  No predictions available. Skipping XAI analysis.

✓ All XAI analyses completed!
Analyzed models: 0

⚠️  No XAI results generated.


# Step 3: Individual Model XAI Visualizations

In [ ]:
print("="*80)
print("STEP 3: INDIVIDUAL MODEL XAI VISUALIZATIONS")
print("="*80)

if not xai_results:
    print("⚠️  No XAI results available for individual visualizations.")
else:
    print("Creating individual XAI visualizations for each model...")

    for model_key, results in xai_results.items():
        arch, resolution = model_key

        if resolution not in sample_images:
            print(f"⚠️  No sample images for {arch} {resolution}px")
            continue

        print(f"Creating visualization for {arch} ({resolution}px)...")

        try:
            plot_individual_xai_results(
                model_key, results,
                sample_images[resolution],
                sample_labels[resolution],
                f"{OUTPUT_PATH}/individual_results"
            )
        except Exception as e:
            print(f"✗ Failed to create visualization for {arch} {resolution}px: {e}")

    print("✓ Individual visualizations completed!")

STEP 3: INDIVIDUAL MODEL XAI VISUALIZATIONS
⚠️  No XAI results available for individual visualizations.


# Step 4: Comprehensive XAI Comparisons

In [ ]:
print("="*80)
print("STEP 4: COMPREHENSIVE XAI COMPARISONS")
print("="*80)

if not xai_results:
    print("⚠️  No XAI results available for comparisons.")
else:
    # 4.1: Compare by Architecture (different resolutions for same architecture)
    print("\n4.1: Comparing by Architecture (Resolution Effects)")
    print("-" * 50)

    try:
        plot_xai_comparison_by_architecture(xai_results, sample_images, sample_labels, f"{OUTPUT_PATH}/comparisons")
    except Exception as e:
        print(f"✗ Architecture comparison failed: {e}")

    # 4.2: Compare by Resolution (different architectures for same resolution)
    print("\n4.2: Comparing by Resolution (Architecture Effects)")
    print("-" * 50)

    try:
        plot_xai_comparison_by_resolution(xai_results, sample_images, sample_labels, f"{OUTPUT_PATH}/comparisons")
    except Exception as e:
        print(f"✗ Resolution comparison failed: {e}")

    print("\n✓ Comparison visualizations completed!")

STEP 4: COMPREHENSIVE XAI COMPARISONS
⚠️  No XAI results available for comparisons.


# Step 5: Misclassification Analysis

In [ ]:
print("="*80)
print("STEP 5: MISCLASSIFICATION ANALYSIS")
print("="*80)

if not predictions_files:
    print("⚠️  No prediction files available for misclassification analysis.")
    misclassification_analysis = {
        'misclassified_by_all': [],
        'correctly_classified_by_all': [],
        'total_samples_analyzed': 0,
        'models_analyzed': []
    }
else:
    # Load all prediction data
    print("Loading prediction data...")
    predictions_data = {}

    for (arch, res), pred_file in predictions_files.items():
        try:
            pred_data = load_model_predictions(pred_file)
            # Use tuple key for consistency with xai_results
            predictions_data[(arch, res)] = pred_data
        except Exception as e:
            print(f"✗ Failed to load {pred_file}: {e}")

    print(f"Loaded predictions for {len(predictions_data)} models")

    # Analyze misclassifications
    print("\nAnalyzing misclassifications...")
    try:
        misclassification_analysis = analyze_misclassifications(
            predictions_data, sample_images, sample_labels, f"{OUTPUT_PATH}/comparisons"
        )

        print(f"Images misclassified by all models: {len(misclassification_analysis['misclassified_by_all'])}")
        print(f"Images correctly classified by all models: {len(misclassification_analysis['correctly_classified_by_all'])}")

    except Exception as e:
        print(f"✗ Misclassification analysis failed: {e}")
        misclassification_analysis = {
            'misclassified_by_all': [],
            'correctly_classified_by_all': [],
            'total_samples_analyzed': 0,
            'models_analyzed': []
        }

print("\n✓ Misclassification analysis completed!")

STEP 5: MISCLASSIFICATION ANALYSIS
⚠️  No prediction files available for misclassification analysis.

✓ Misclassification analysis completed!


# Step 6: XAI Similarity Metrics

In [ ]:
print("="*80)
print("STEP 6: XAI SIMILARITY METRICS")
print("="*80)

if not xai_results:
    print("⚠️  No XAI results available for similarity metrics.")
    similarity_metrics = {}
else:
    # Calculate similarity metrics between XAI methods
    print("Calculating XAI similarity metrics...")
    try:
        similarity_metrics = calculate_xai_similarity_metrics(xai_results)

        print("\nXAI Method Similarity Results:")
        print("-" * 40)

        for model_key, metrics in similarity_metrics.items():
            print(f"\n{model_key}:")
            for method_pair, values in metrics.items():
                print(f"  {method_pair}:")
                for metric_name, value in values.items():
                    if not np.isnan(value):
                        print(f"    {metric_name}: {value:.3f}")
                    else:
                        print(f"    {metric_name}: N/A")

    except Exception as e:
        print(f"✗ Similarity metrics calculation failed: {e}")
        similarity_metrics = {}

print("\n✓ Similarity metrics calculated!")

STEP 6: XAI SIMILARITY METRICS
⚠️  No XAI results available for similarity metrics.

✓ Similarity metrics calculated!


# Step 7: Generate Comprehensive Analysis Report

In [ ]:
print("="*80)
print("STEP 7: GENERATING COMPREHENSIVE ANALYSIS REPORT")
print("="*80)

# Generate and save comprehensive report
print("Generating comprehensive analysis report...")

if not xai_results:
    print("⚠️  No XAI results available. Creating minimal report.")
    analysis_report = {
        'summary': {
            'total_models_analyzed': 0,
            'xai_methods': ['GradCAM', 'LIME', 'SHAP'],
            'total_misclassifications': 0,
            'total_correct_classifications': 0
        },
        'model_summaries': {},
        'similarity_analysis': {},
        'misclassification_analysis': misclassification_analysis
    }
else:
    try:
        analysis_report = save_xai_analysis_report(
            xai_results, similarity_metrics, misclassification_analysis, OUTPUT_PATH
        )
    except Exception as e:
        print(f"✗ Report generation failed: {e}")
        analysis_report = {'error': str(e)}

if 'model_summaries' in analysis_report and analysis_report['model_summaries']:
    print("\nANALYSIS SUMMARY:")
    print("="*50)

    # Model Performance Summary
    print("\nModel Performance Summary:")
    print("-" * 30)
    for model_key, summary in analysis_report['model_summaries'].items():
        print(f"\n{model_key}:")
        print(f"  Accuracy: {summary['model_accuracy']:.3f}")
        print(f"  GradCAM Success: {summary['gradcam_success_rate']:.3f}")
        print(f"  LIME Success: {summary['lime_success_rate']:.3f}")
        print(f"  SHAP Success: {summary['shap_success_rate']:.3f}")

    # XAI Method Comparison Summary
    print(f"\nXAI Method Effectiveness:")
    print("-" * 30)
    all_gradcam_success = [s['gradcam_success_rate'] for s in analysis_report['model_summaries'].values()]
    all_lime_success = [s['lime_success_rate'] for s in analysis_report['model_summaries'].values()]
    all_shap_success = [s['shap_success_rate'] for s in analysis_report['model_summaries'].values()]

    if all_gradcam_success:
        print(f"GradCAM: {np.mean(all_gradcam_success):.3f} ± {np.std(all_gradcam_success):.3f} success rate")
    if all_lime_success:
        print(f"LIME: {np.mean(all_lime_success):.3f} ± {np.std(all_lime_success):.3f} success rate")
    if all_shap_success:
        print(f"SHAP: {np.mean(all_shap_success):.3f} ± {np.std(all_shap_success):.3f} success rate")

    # Similarity Analysis Summary
    if similarity_metrics:
        print(f"\nXAI Method Similarity (where available):")
        print("-" * 40)
        all_cosine_similarities = []
        all_pearson_correlations = []

        for model_metrics in similarity_metrics.values():
            for method_pair, values in model_metrics.items():
                if not np.isnan(values['mean_cosine']):
                    all_cosine_similarities.append(values['mean_cosine'])
                if not np.isnan(values['mean_pearson']):
                    all_pearson_correlations.append(values['mean_pearson'])

        if all_cosine_similarities:
            print(f"Average Cosine Similarity: {np.mean(all_cosine_similarities):.3f} ± {np.std(all_cosine_similarities):.3f}")
        if all_pearson_correlations:
            print(f"Average Pearson Correlation: {np.mean(all_pearson_correlations):.3f} ± {np.std(all_pearson_correlations):.3f}")

print(f"\n✓ Comprehensive analysis completed!")
print(f"📁 All results saved to: {OUTPUT_PATH}")
print(f"📊 Individual results: {OUTPUT_PATH}/individual_results/")
print(f"📊 Comparison plots: {OUTPUT_PATH}/comparisons/")
print(f"🔍 Analysis report: {OUTPUT_PATH}/xai_analysis_report.json")

STEP 7: GENERATING COMPREHENSIVE ANALYSIS REPORT
Generating comprehensive analysis report...
⚠️  No XAI results available. Creating minimal report.

✓ Comprehensive analysis completed!
📁 All results saved to: ../output/xai_results/
📊 Individual results: ../output/xai_results//individual_results/
📊 Comparison plots: ../output/xai_results//comparisons/
🔍 Analysis report: ../output/xai_results//xai_analysis_report.json


# Analysis Complete

**All XAI analysis steps have been completed successfully!**

Check the output directories for detailed results and visualizations.

In [ ]:
print("\n" + "="*80)
print("ANALYSIS COMPLETED - CHECK OUTPUT FOLDERS FOR DETAILED RESULTS")
print("="*80)
print(f"📁 All results saved to: {OUTPUT_PATH}")
print(f"📊 Comparison plots saved to: {OUTPUT_PATH}/comparisons/")
print(f"🔍 Analysis report: {OUTPUT_PATH}/xai_analysis_report.json")


ANALYSIS COMPLETED - CHECK OUTPUT FOLDERS FOR DETAILED RESULTS
📁 All results saved to: ../output/xai_results/
📊 Comparison plots saved to: ../output/xai_results//comparisons/
🔍 Analysis report: ../output/xai_results//xai_analysis_report.json
